In [ ]:
import feedparser
import openai
import datetime
from dateutil import parser as dateparser

# ------------- CONFIG ----------------
openai.api_key = "YOUR_API_KEY"   # <-- put your OpenAI key here

today = datetime.date.today()           # get current date
date_str = today.strftime("%Y-%m-%d")

# RSS feeds (you can add more)
RSS_FEEDS = {
    "NEP-ECM": "https://nep.repec.org/nep-ecm.rdf",   # Econometrics
    "NEP-MAC": "https://nep.repec.org/nep-mac.rdf",   # Macroeconomics
    "NEP-ENE": "https://nep.repec.org/nep-ene.rdf",   # Energy
    "arXiv-econEM": "http://export.arxiv.org/rss/econ.EM",
    "arXiv-statML": "http://export.arxiv.org/rss/stat.ML"
}

SCORE_THRESHOLD = 7.0  # keep only papers scoring above this
DAYS_BACK = 7          # look back 7 days
OUTPUT_FILE = f"econ4_{date_str}.md"

# ------------- FUNCTIONS ----------------

def fetch_recent_papers():
    """Fetch recent papers (last DAYS_BACK) from RSS feeds."""
    cutoff_date = datetime.datetime.utcnow() - datetime.timedelta(days=DAYS_BACK)
    papers = []

    for source, url in RSS_FEEDS.items():
        feed = feedparser.parse(url)
        for entry in feed.entries:
            # Extract date if available
            if hasattr(entry, "published"):
                pub_date = dateparser.parse(entry.published)
            elif hasattr(entry, "updated"):
                pub_date = dateparser.parse(entry.updated)
            else:
                continue

            if pub_date < cutoff_date:
                continue

            papers.append({
                "title": entry.title,
                "link": entry.link,
                "summary": getattr(entry, "summary", ""),
                "authors": getattr(entry, "author", "Unknown"),
                "source": source,
                "date": pub_date.strftime("%Y-%m-%d")
            })
    return papers

def score_and_summarize(paper):
    """Send abstract to LLM for summary and scoring."""
    prompt = f"""
You are helping curate a newsletter about novel forecasting methods in economics.
Here is a paper:

Title: {paper['title']}
Authors: {paper['authors']}
Source: {paper['source']}
Date: {paper['date']}
Abstract: {paper['summary']}

Task:
1. Summarize the paper in 2–3 sentences.
2. Give a relevance score from 1–10 for economic forecasting (consider novelty, methodological contribution, and potential applications).

Return in JSON:
{{
  "summary": "...",
  "score": number
}}
"""

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",   # or "gpt-4.1-mini"
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    try:
        import json
        content = response.choices[0].message["content"]
        data = json.loads(content)
        return data
    except Exception:
        return {"summary": "Error parsing summary.", "score": 0}

def build_markdown(papers):
    """Create markdown file with shortlisted papers."""
    today = datetime.datetime.today().strftime("%Y-%m-%d")
    lines = [f"# EconForecasting Weekly – {today}\n", "## Candidate Papers\n"]

    for p in papers:
        lines.append(f"### [{p['title']}]({p['link']})")
        lines.append(f"- Authors: {p['authors']}")
        lines.append(f"- Source: {p['source']} ({p['date']})")
        lines.append(f"- Score: {p['score']}/10")
        lines.append(f"- Summary: {p['summary']}\n")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✅ Markdown file saved: {OUTPUT_FILE}")

# ------------- MAIN ----------------

if __name__ == "__main__":
    print("Fetching papers...")
    papers = fetch_recent_papers()
    print(f"Found {len(papers)} recent papers.")

    shortlisted = []
    for paper in papers:
        result = score_and_summarize(paper)
        paper.update(result)
        if paper["score"] >= SCORE_THRESHOLD:
            shortlisted.append(paper)

    build_markdown(shortlisted)
    print(f"Shortlisted {len(shortlisted)} papers (score ≥ {SCORE_THRESHOLD}).")
